### Structured Output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LnagChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FF12AD69F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FF12AD61E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The year the movie was released")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")


In [3]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured


_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FF12AD69F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FF12AD61E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'release_year': {'description': 'The year the mov

In [4]:
model.invoke("Provide details about the movie Inception.")


AIMessage(content='<think>\nOkay, so I need to figure out details about the movie Inception. Let me start by recalling what I know. It\'s a 2010 movie directed by Christopher Nolan, right? The main character is Dom Cobb, played by Leonardo DiCaprio. The movie involves dreams within dreams, using technology to enter people\'s minds. There\'s a concept called "inception," which is planting an idea in someone\'s subconscious. The plot is pretty complex, with layers of reality and time dilation depending on the dream level.\n\nI should mention the main cast: DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. The themes include reality vs. dreams, guilt, and the mind\'s power. The director is Christopher Nolan, known for other films like The Dark Knight and Interstellar. The cinematography is by Wally Pfister, and there\'s a lot of visual effects, like the rotating hallway fight scene.\n\nThe plot summary: Cobb\'s team enters the subconscious of a target to plant an idea. Th

### Message output alongside Parsed Structure

In [5]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A model representing a movie with its details.
    """
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The year the movie was released")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")

model_with_structured = model.with_structured_output(Movie, include_raw=True)

response = model_with_structured.invoke("Provide details about the movie Inception.")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There\'s a Movie function with parameters like director, rating, release_year, title, and year. All these fields are required. I need to fill in the correct information for Inception. The director is Christopher Nolan. The release year was 2010. The rating is probably around 8.8 on IMDb. The title is "Inception". I should make sure the year and release_year are the same, both 2010. Let me confirm the rating. Yes, IMDb gives it 8.8. So I\'ll structure the tool call with these details. Need to present it in JSON within the tool_call tags.\n', 'tool_calls': [{'id': 'fct8zn5v9', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"release_year":2010,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 211, 'prompt_tokens': 265,

### Nested Structure

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    """A model representing an actor with their details.
    """
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")
    birth_year: int = Field(..., description="The birth year of the actor")

class MovieDetails(BaseModel):
    """A model representing a movie with its details and cast.
    """
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The year the movie was released")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")
    cast: list[Actor] = Field(..., description="The list of actors in the mnnovie")
    genres: list[str] = Field(..., description="The genres of the movie")
    budget: int = Field(..., description="The budget of the movie in USD")
    

In [7]:
model_with_structured = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structured.invoke("Provide details about the movie Inception, including its cast, genres, and budget.")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception, specifically the cast, genres, and budget. Let me check the available functions. There's a MovieDetails function that includes all those parameters. I need to make sure to call that function with the correct arguments. Let me structure the response by gathering the necessary information. The cast should be a list of actors with their names, roles, and birth years. The genres are an array of strings. The budget is an integer. I'll need to format all that into the function call properly. Let me verify the required fields to ensure nothing is missing.\n", 'tool_calls': [{'id': '3msyag0hf', 'function': {'arguments': '{"budget":160000000,"cast":[{"birth_year":1974,"name":"Leonardo DiCaprio","role":"Cobb"},{"birth_year":1979,"name":"Joseph Gordon-Levitt","role":"Arthur"},{"birth_year":1987,"name":"Elliot Page","role":"Ariadne"},{"birth_year":1977,"name":"Tom 